# Data separation: split into train / test

Source: `data/720*480/{original,edited}` (paired by identical filename).

Creates:
```
data/720*480/train/original   data/720*480/train/edited
data/720*480/test/original    data/720*480/test/edited
```

The split is decided **per filename (per pair)**: each pair's original *and* edited go to the same split, so train and test stay one-to-one (every original has its edited, and vice versa). A fixed seed makes the split reproducible, and files are **copied** so the source folder is untouched.

In [ ]:
import os
import random
import shutil

# Paths assume this notebook lives in <project>/code/
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
SRC_DIR = os.path.join(DATA_DIR, "720*480")
SPLIT_DIR = os.path.join(DATA_DIR, "720*480")
FOLDERS = ["original", "edited"]

# Config
TRAIN_RATIO = 0.8   # 80% train, 20% test
SEED = 42           # reproducible shuffle

for f in FOLDERS:
    p = os.path.join(SRC_DIR, f)
    print(f, "->", p, "| exists:", os.path.isdir(p))

In [ ]:
# Build the list of paired filenames (present in BOTH original and edited).
o_files = set(f for f in os.listdir(os.path.join(SRC_DIR, "original")) if f.lower().endswith(".jpg"))
e_files = set(f for f in os.listdir(os.path.join(SRC_DIR, "edited")) if f.lower().endswith(".jpg"))
pairs = sorted(o_files & e_files)

only_o = o_files - e_files
only_e = e_files - o_files
print("paired files:", len(pairs))
print("only in original (ignored):", len(only_o))
print("only in edited (ignored):", len(only_e))

# Shuffle and split per pair.
random.seed(SEED)
shuffled = pairs[:]
random.shuffle(shuffled)
n_train = int(len(shuffled) * TRAIN_RATIO)
train_files = sorted(shuffled[:n_train])
test_files = sorted(shuffled[n_train:])
print(f"train: {len(train_files)} | test: {len(test_files)}")

In [ ]:
# Copy each pair into its split. Both original and edited follow the same split.
def copy_split(file_list, split_name):
    for folder in FOLDERS:
        dst = os.path.join(SPLIT_DIR, split_name, folder)
        if os.path.isdir(dst):
            shutil.rmtree(dst)      # clear stale files to avoid cross-split duplicates
        os.makedirs(dst, exist_ok=True)
        for f in file_list:
            shutil.copy2(os.path.join(SRC_DIR, folder, f), os.path.join(dst, f))
    print(f"{split_name}: copied {len(file_list)} pairs")

copy_split(train_files, "train")
copy_split(test_files, "test")

In [ ]:
# VERIFY: within each split original/edited match 1:1, and train/test do not overlap.
def names(split, folder):
    return set(os.listdir(os.path.join(SPLIT_DIR, split, folder)))

for split in ["train", "test"]:
    o, e = names(split, "original"), names(split, "edited")
    print(f"{split}: original={len(o)} edited={len(e)} | match 1:1: {o == e}")

overlap = names("train", "original") & names("test", "original")
print("train/test overlap (should be 0):", len(overlap))